# Exercises: Bayesian Networks

For each of the following situations, propose a suitable node ordering, select a suitable set of parents for each node, and draw the graph. Suggest a plausible conditional probability distribution for each node. Then, for each situation:

* count the number of free parameters in your network and compare it with the number needed for the full joint distribution;
* implement the conditional probability tables as numpy arrays, using the naming convention of the notes (for example, $P(A\vert B,C)$ is `PA_BC`) and check that each one is correctly normalised;
* build the joint distribution with `numpy.einsum`, and use it to find the probability of the *first-named* cause given that the effect has been observed. Does the probability change as you would expect when you observe a second variable? (The helper functions `draw_dag`, `cpt` and `posterior` below may be used; see the notes for how they work.)

1. Cooking dinner, house on fire, smoke alarm

2. Heavy rain, burst water pipe, flooded patio, call plumber

3. Late for train, speeding, speeding ticket, camera malfunction

4. Sprinkler, rain, wet grass

5. Car alarm sounding, earthquake, heavy traffic, attempted car theft, angry neighbours

6. Puncture, fuel pump failure, recovery vehicle, large bill, late for work

7. Show that the graph for situation 3 contains a collider. Which two variables become dependent when the collider is observed? Verify this numerically for your network.

8. Two different people draw networks for situation 4. One draws $R\to S\to W$ and $R\to W$; the other draws $R\to W\leftarrow S$ with no edge between $R$ and $S$. Which is more plausible and why? How many parameters does each need?

In [1]:
import numpy as np

import html as _html
from pyvis.network import Network      # pip install pyvis
from IPython.display import HTML, display

def draw_dag(pos, edges, title=None, scale=110, cdn_resources='remote'):
    """Draw a DAG interactively with pyvis (the nodes can be dragged with the mouse).
    pos: {node: (x, y)} with y increasing upwards, edges: list of (parent, child).
    cdn_resources='remote' loads the vis.js library from the internet; use 'in_line' to embed it (offline use)."""
    xs, ys = zip(*pos.values())
    height = int((max(ys) - min(ys)) * scale + 140)
    net = Network(height=f"{height}px", width="100%", directed=True, notebook=True,
                  cdn_resources=cdn_resources)
    for n, (x, y) in pos.items():
        net.add_node(n, label=n, x=x*scale, y=-y*scale, physics=False, shape="circle",
                     color={"background": "#E6F1FB", "border": "#378ADD"},
                     font={"size": 20, "color": "#042C53"}, margin=12)
    for parent, child in edges:
        net.add_edge(parent, child, color="#555555", width=2, arrows="to")
    net.set_options('{"physics": {"enabled": false}, "edges": {"smooth": false}}')
    page = net.generate_html(notebook=True)
    heading = f"<p><b>{title}</b></p>" if title else ""
    display(HTML(f'<div>{heading}<iframe srcdoc="{_html.escape(page, quote=True)}" width="100%" '
                 f'height="{height + 40}" style="border:0"></iframe></div>'))

def cpt(p1):
    """Conditional probability table for a binary variable from the array p1 = P(X=1 | parents).
    Returns an array indexed [x][parents...] with [0] = P(X=0|...) and [1] = P(X=1|...)."""
    p1 = np.asarray(p1, dtype=float)
    return np.array([1 - p1, p1])

def posterior(PJ, names, query, evidence={}):
    """P(query | evidence) from a joint distribution PJ with one axis per variable in `names`.
    evidence is a dictionary {variable name: value}."""
    index = tuple(evidence.get(n, slice(None)) for n in names)
    sub = PJ[index]
    remaining = [n for n in names if n not in evidence]
    other_axes = tuple(i for i, n in enumerate(remaining) if n != query)
    p = sub.sum(axis=other_axes)
    return p / p.sum()